In [6]:
import pandas as pd
from pacti.contracts import PolyhedralIoContract

#to create contracts we need the IL IH: 
#structure should be as follows for a NOR gate:
# assume (x < IL)
# then gaurentee output = ON 
#assume (x > IH)
# then gaurentee output = OFF
#assume (IL < x < IH) 
# then gaurentee output = transition 

df = pd.read_csv("/home/shrut/bioCircuitProject/library/NOR_gate/parsed_Eco1C1G1T1.csv")

#to be able to figure out corresponding y values for given x values (IL and IH are x values)
#also important because contracts say assuming x is in a certain range, then y is guaranteed to be in a certain range 
def response(x, row):
    ymin = float(row["ymin"])
    ymax = float(row["ymax"])
    k = float(row["K"])
    n = float(row["n"])
    return ymin + (ymax - ymin) / (1.0 + (x / k) ** n)


def make_nor_contracts(row):
    #contracts are based on these two x - values 
    il = float(row["IL"])
    ih = float(row["IH"])

    #need corresponding y values for the regions defined by IL and IH (for the gaurentee part of the contract)

    y_at_il = response(il, row)
    y_at_ih = response(ih, row)
    ymin = float(row["ymin"])
    ymax = float(row["ymax"])

   
    def number(value):
        return f"{value:.12g}"

    # [ymin, y(IL)] -> range of y values for: x < IL -> OFF
    # [y(IL), y(IH)] -> range of y values for: IL < x < IH -> transition
    # [y(IH), ymax] -> range of y values for: x > IH -> ON

    return {
        "ON": PolyhedralIoContract.from_strings(
            input_vars=["x"],
            output_vars=["y"],

            assumptions=["-x <= 0", f"x <= {il:.12g}"], #negating both sides reverses ineqality 
            guarantees=[f"-y <= {-y_at_il:.12g}", f"y <= {number(ymax)}"],
        ),
        "transition": PolyhedralIoContract.from_strings(
            input_vars=["x"],
            output_vars=["y"],
            assumptions=[f"-x <= {-il:.12g}", f"x <= {number(ih)}"],
            guarantees=[f"-y <= {-y_at_ih:.12g}", f"y <= {number(y_at_il)}"],
        ),
        "OFF": PolyhedralIoContract.from_strings(
            input_vars=["x"],
            output_vars=["y"],
            assumptions=[f"-x <= {-ih:.12g}"],
            guarantees=[f"-y <= {-ymin:.12g}", f"y <= {number(y_at_ih)}"],
        ),
    }

contracts_by_gate = {}
for i, row in df.iterrows():
    protein_name = str(row["name"])
    contracts_by_gate[protein_name] = make_nor_contracts(row)


# for i in range(len(df)):
#     protein_name = str(df.iloc[i]["name"])
#     contracts_by_gate[protein_name]["ON"]
#     contracts_by_gate[protein_name]["OFF"]
#     contracts_by_gate[protein_name]["transition"]